# 03: NHANES and UCI preparation

Design decisions
- NHANES 2015-2016 and 2017-2018 (14 files per cycle). The notebook looks for the files anywhere under datasets/ and downloads the missing ones from CDC into datasets/raw/nhanes/. If Colab cannot reach CDC, it stops at the first failure and prints every link to download by hand.
- Population: adults aged 20 and over. Outcome: ever told coronary heart disease (MCQ160C) or a heart attack (MCQ160E), built like BRFSS _MICHD.
- Predictors: the 16 BRFSS indicators NHANES can express (days of poor physical or mental health, "could not see a doctor because of cost", fruit and vegetables are not available). Education is coarsened to 5 levels and income to 7, and the same coarsening is applied to BRFSS. HighChol is defined only for people whose cholesterol was ever checked, as BRFSS asks it. Every rule is in results/03_nhanes_variable_map.csv; any undocumented code stops the notebook.
- Measured values for the measurement shift: examination BMI; high blood pressure from the mean of the examination readings (140/90); total cholesterol 240 mg/dL or more. Complete cases on the predictors, the outcome, measured BMI and measured blood pressure, so the self-reported and measured versions describe the same people.
- UCI Heart Disease (four sites, downloaded by notebook 01): eight predictors recorded at every site, zero blood pressure or cholesterol treated as missing, complete cases. Missingness per site is written out.
- Outcome definitions for every dataset are frozen in results/03_outcome_map.csv.
- Outputs on Drive: datasets/derived/nhanes/nhanes_2015_2018.parquet and datasets/derived/uci/uci_heart_sites.parquet.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyreadstat'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, os, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public
try:
    from scax.data import nhanes as nh
    from scax.data import uci
    from scax.data.outcomes import OUTCOME_MAP
except ImportError:
    raise RuntimeError('The stage 10 source is not loaded. Run the stage 10 batch, then Runtime > Disconnect and delete runtime, then run again.')

seed_everything(42)
NH_RAW = paths.RAW / 'nhanes'; NH_RAW.mkdir(parents=True, exist_ok=True)
NH_OUT = paths.DERIVED / 'nhanes'; NH_OUT.mkdir(parents=True, exist_ok=True)
UCI_OUT = paths.DERIVED / 'uci'; UCI_OUT.mkdir(parents=True, exist_ok=True)

FOUND = {}
for root, dirs, names in os.walk(paths.DATASETS):
    dirs[:] = [d for d in dirs if d != 'derived']
    for n in sorted(names):
        k = nh.stem_of(n)
        if k is not None and k not in FOUND:
            FOUND[k] = Path(root) / n
WANTED = [(c, y) for y in nh.CYCLES for c in nh.COMPONENTS]
missing = [k for k in WANTED if k not in FOUND]
print(f'NHANES files found on Drive: {len(WANTED) - len(missing)} of {len(WANTED)}')
failure = None
for c, y in missing:
    dest = NH_RAW / f'{nh.file_stem(c, y)}.xpt'
    try:
        req = urllib.request.Request(nh.url(c, y), headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=60) as r:
            data = r.read()
        if not data.startswith(b'HEADER RECORD'):
            raise ValueError('the server did not return a SAS transport file')
        dest.write_bytes(data)
        FOUND[(c, y)] = dest
        print(f'  downloaded {dest.name} ({len(data) / 1e6:.1f} MB)', flush=True)
    except Exception as e:
        failure = f'{type(e).__name__}: {e}'
        break
still = [k for k in WANTED if k not in FOUND]
if still:
    print(f'Colab could not download from CDC ({failure}).')
    print(f'Download these {len(still)} files in a browser and put them in {NH_RAW}, then run this notebook again:')
    for c, y in still:
        print('  ', nh.url(c, y))
    raise RuntimeError('NHANES files missing (links above)')

## NHANES in the BRFSS coding

In [ ]:
TABLES, DROPS = {}, []
for year in nh.CYCLES:
    t0 = time.time()
    frames = {c: nh.read_component(FOUND[(c, year)], c, year) for c in nh.COMPONENTS}
    merged = nh.merge(frames)
    t, d = nh.harmonise(merged, year)
    d.insert(0, 'cycle', f'{year}-{year + 1}')
    TABLES[year] = t
    DROPS.append(d)
    print(f'{year}-{year + 1}: {len(merged):,} persons, {len(t):,} adults kept, heart disease {t[nh.TARGET].mean():.2%}, '
          f'{time.time() - t0:.0f}s')
NH = pd.concat(TABLES.values(), ignore_index=True)
DROP = pd.concat(DROPS, ignore_index=True)
NH_FILE = NH_OUT / 'nhanes_2015_2018.parquet'
NH.to_parquet(NH_FILE, index=False)
pd.set_option('display.width', 250)
print(DROP.pivot_table(index='step', columns='cycle', values='records', sort=False).to_string())

In [ ]:
# self-reported against measured values (the measurement shift), and NHANES against BRFSS 2015
def agreement(self_col, meas_col, data):
    d = data.dropna(subset=[self_col, meas_col])
    a, b = d[self_col].astype(int), d[meas_col].astype(int)
    return {'records': int(len(d)), 'self_reported_yes': float(a.mean()), 'measured_yes': float(b.mean()),
            'both_yes': float(((a == 1) & (b == 1)).mean()), 'self_only': float(((a == 1) & (b == 0)).mean()),
            'measured_only': float(((a == 0) & (b == 1)).mean()), 'agreement': float((a == b).mean()),
            'kappa': float(cohen_kappa_score(a, b))}

AGREE = {'high blood pressure': agreement('HighBP', 'HighBP_measured', NH),
         'high cholesterol': agreement('HighChol', 'HighChol_measured', NH)}
diff = NH.BMI_self_exact - NH.BMI_measured_exact
AGREE['BMI'] = {'records': int(diff.notna().sum()), 'mean_self_minus_measured': float(diff.mean()),
                'sd': float(diff.std()), 'share_rounded_equal': float((NH.BMI == NH.BMI_measured).mean()),
                'obese_self_reported': float((NH.BMI_self_exact >= 30).mean()),
                'obese_measured': float((NH.BMI_measured_exact >= 30).mean())}
print(json.dumps(AGREE, indent=1))

REC02 = json.load(open(f'{paths.RESULTS}/02_brfss_rebuild.json'))
f15 = paths.DERIVED / 'brfss' / 'brfss_2015.parquet'
assert sha256_file(f15) == REC02['parquet_sha256']['2015'], 'brfss_2015.parquet differs from the file notebook 02 wrote'
B15 = pd.read_parquet(f15)
BC = nh.brfss_common(B15)
COMPARE = pd.DataFrame({'BRFSS 2015': BC.mean(), 'NHANES': NH[nh.N_PREDICTORS].mean()})
for k, v in nh.MEASURED.items():
    COMPARE.loc[f'{k} (measured)', 'NHANES'] = NH[v].mean()
COMPARE.loc[nh.TARGET] = [B15[nh.TARGET].mean(), NH[nh.TARGET].mean()]
COMPARE.loc['records'] = [len(B15), len(NH)]
print(COMPARE.round(3).to_string())

## UCI Heart Disease sites

In [ ]:
XU, yU = public.LOADERS['uci_heart'](paths.RAW)
MISS = uci.missingness(XU)
UCI_T, UCI_COUNTS = uci.prepare(XU, yU)
UCI_FILE = UCI_OUT / 'uci_heart_sites.parquet'
UCI_T.to_parquet(UCI_FILE, index=False)
print('missing share per site (zero blood pressure or cholesterol counted as missing):')
print(MISS.round(2).to_string())
print(UCI_COUNTS.round(3).to_string(index=False))

## Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
SELF, MEAS = '#0072B2', '#D55E00'
fig, axes = plt.subplots(1, 3, figsize=(7.4, 2.7))
ax = axes[0]
ax.hist(diff.clip(-10, 10), bins=np.arange(-10, 10.5, 0.5), color='0.55', edgecolor='white', linewidth=0.3)
ax.axvline(0, color='0.2', lw=0.8, ymax=0.8)
ax.axvline(diff.median(), color=MEAS, lw=1.2, ls='--', ymax=0.8, label=f'median {diff.median():+.1f}')
ax.set_xlabel('BMI: self-reported minus measured (kg/m²)')
ax.set_ylabel('Adults')
ax.set_ylim(top=ax.get_ylim()[1] * 1.2)
ax.legend(frameon=False, fontsize=7, loc='upper left')
ax.set_title('BMI', fontsize=8.5)
bands = pd.cut(NH.age_years, [19, 39, 59, 200], labels=['20-39', '40-59', '60+'])
for ax, (s, m, title) in zip(axes[1:], [('HighBP', 'HighBP_measured', 'High blood pressure'),
                                        ('HighChol', 'HighChol_measured', 'High cholesterol')]):
    g = NH.assign(band=bands).dropna(subset=[m]).groupby('band', observed=True)[[s, m]].mean()
    x = np.arange(len(g))
    ax.bar(x - 0.2, g[s], 0.38, color=SELF, label='Ever told (self-reported)')
    ax.bar(x + 0.2, g[m], 0.38, color=MEAS, label='Measured at examination')
    ax.set_xticks(x, list(g.index))
    ax.set_xlabel('Age (years)')
    ax.set_axisbelow(True)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_title(title, fontsize=8.5)
    ax.grid(axis='y', color='0.92', lw=0.6)
axes[1].set_ylabel('Share of adults')
h, l = axes[1].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, fontsize=7, bbox_to_anchor=(0.62, 1.04))
fig.tight_layout(rect=(0, 0, 1, 0.92))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_nhanes_measurement.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
pd.DataFrame(nh.VARIABLE_MAP, columns=['indicator', 'brfss_item', 'nhanes_2015_2016', 'nhanes_2017_2018', 'rule',
                                        'comparability']).to_csv(f'{paths.RESULTS}/03_nhanes_variable_map.csv', index=False)
pd.DataFrame(OUTCOME_MAP).to_csv(f'{paths.RESULTS}/03_outcome_map.csv', index=False)
DROP.to_csv(f'{paths.RESULTS}/03_nhanes_drop_counts.csv', index=False)
COMPARE.rename_axis('variable').to_csv(f'{paths.RESULTS}/03_nhanes_vs_brfss.csv')
MISS.rename_axis('site').to_csv(f'{paths.RESULTS}/03_uci_missingness.csv')
UCI_COUNTS.to_csv(f'{paths.RESULTS}/03_uci_sites.csv', index=False)
SUM = NH.groupby('cycle').agg(records=(nh.TARGET, 'size'), heart_disease=(nh.TARGET, 'mean')).reset_index()
write_json(f'{paths.RESULTS}/03_nhanes_uci.json', {
    'nhanes': {'cycles': {f'{y}-{y + 1}': nh.CYCLES[y] for y in nh.CYCLES}, 'components': nh.COMPONENTS,
               'predictors': nh.N_PREDICTORS, 'measured': nh.MEASURED, 'records': int(len(NH)),
               'per_cycle': SUM.to_dict('records'), 'heart_disease_share': float(NH[nh.TARGET].mean()),
               'self_vs_measured': AGREE, 'parquet_sha256': sha256_file(NH_FILE),
               'raw_file_sha256': {nh.file_stem(c, y): sha256_file(FOUND[(c, y)]) for c, y in WANTED},
               'brfss_2015_parquet_sha256': REC02['parquet_sha256']['2015']},
    'uci': {'predictors': uci.PREDICTORS, 'left_out': uci.LEFT_OUT, 'sites': UCI_COUNTS.to_dict('records'),
            'parquet_sha256': sha256_file(UCI_FILE)}})
print('written: results/03_nhanes_variable_map.csv, 03_outcome_map.csv, 03_nhanes_drop_counts.csv, 03_nhanes_vs_brfss.csv, '
      '03_uci_missingness.csv, 03_uci_sites.csv, 03_nhanes_uci.json; figures/fig_nhanes_measurement')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "03: NHANES 2015-2018 in the BRFSS coding (16 common predictors, measured BMI, blood pressure, cholesterol), UCI sites prepared, outcome definitions frozen, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)